# ResNet50 for ISUP Grade Classification

This notebook tests ResNet50 architecture for prostate cancer grading.

In [8]:
from torch import optim
from torchvision.models import resnet50, ResNet50_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
import sys
sys.path.append('../../..')
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint, evaluation, format_metrics
from utils.train import train_model

In [9]:
seed = 42
batch_size = 2
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


## ResNet50 Model Wrapper

In [10]:
class ResNetClassifier(nn.Module):
    """ResNet50 wrapper for ordinal classification"""
    def __init__(self, model, output_dimensions, dropout_rate=0.5):
        super(ResNetClassifier, self).__init__()
        self.model = model
        
        # Get number of features from fc layer
        num_features = model.fc.in_features
        
        # Remove original fc layer
        self.model.fc = nn.Identity()
        
        # Create new classifier
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(num_features, output_dimensions)
        )
    
    def forward(self, x):
        features = self.model(x)
        output = self.classifier(features)
        return output

In [11]:
# Load pretrained ResNet50
base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model = ResNetClassifier(model=base_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 23,518,277
Trainable parameters: 23,518,277


## Load Dataset

In [12]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]

df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

Train: (7219, 5), Val: (1805, 5), Test: (1592, 4)


In [13]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)

## Training

In [14]:
optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/resnet50.txt",
    path_to_save_model="models/resnet50.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 903/903 [03:22<00:00,  4.47it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.445
VAL_ACC      Mean: 43.479 | Std: 1.181 | 95% CI: [41.551, 45.540]
VAL_KAPPA    Mean: 0.693 | Std: 0.015 | 95% CI: [0.670, 0.716]
VAL_F1       Mean: 0.397 | Std: 0.012 | 95% CI: [0.378, 0.416]
VAL_RECALL   Mean: 0.391 | Std: 0.012 | 95% CI: [0.373, 0.411]
VAL_PRECISION Mean: 0.433 | Std: 0.013 | 95% CI: [0.411, 0.455]
Salvando o melhor modelo... 0.6858557582688711 -> 0.6932258757098662
Epoch 3/50



100%|██████████| 903/903 [03:23<00:00,  4.44it/s]


VAL_LOSS     0.457
VAL_ACC      Mean: 48.631 | Std: 1.179 | 95% CI: [46.648, 50.582]
VAL_KAPPA    Mean: 0.740 | Std: 0.013 | 95% CI: [0.718, 0.761]
VAL_F1       Mean: 0.444 | Std: 0.012 | 95% CI: [0.423, 0.463]
VAL_RECALL   Mean: 0.442 | Std: 0.012 | 95% CI: [0.422, 0.461]
VAL_PRECISION Mean: 0.480 | Std: 0.014 | 95% CI: [0.458, 0.502]
Salvando o melhor modelo... 0.6932258757098662 -> 0.7398321813738816
Epoch 4/50



...

VAL_LOSS     1.046
VAL_ACC      Mean: 52.682 | Std: 1.181 | 95% CI: [50.859, 54.626]
VAL_KAPPA    Mean: 0.707 | Std: 0.015 | 95% CI: [0.681, 0.732]
VAL_F1       Mean: 0.426 | Std: 0.012 | 95% CI: [0.408, 0.446]
VAL_RECALL   Mean: 0.428 | Std: 0.010 | 95% CI: [0.412, 0.446]
VAL_PRECISION Mean: 0.458 | Std: 0.014 | 95% CI: [0.436, 0.482]
Epoch 16/50



100%|██████████| 903/903 [03:10<00:00,  4.73it/s]


VAL_LOSS     1.002
VAL_ACC      Mean: 54.682 | Std: 1.192 | 95% CI: [52.742, 56.676]
VAL_KAPPA    Mean: 0.754 | Std: 0.013 | 95% CI: [0.732, 0.776]
VAL_F1       Mean: 0.462 | Std: 0.012 | 95% CI: [0.443, 0.483]
VAL_RECALL   Mean: 0.460 | Std: 0.011 | 95% CI: [0.442, 0.479]
VAL_PRECISION Mean: 0.485 | Std: 0.014 | 95% CI: [0.463, 0.508]
Epoch 17/50



100%|██████████| 903/903 [03:10<00:00,  4.73it/s]


VAL_LOSS     0.964
VAL_ACC      Mean: 54.777 | Std: 1.189 | 95% CI: [52.798, 56.731]
VAL_KAPPA    Mean: 0.756 | Std: 0.013 | 95% CI: [0.734, 0.778]
VAL_F1       Mean: 0.473 | Std: 0.012 | 95% CI: [0.453, 0.494]
VAL_RECALL   Mean: 0.468 | Std: 0.011 | 95% CI: [0.448, 0.487]
VAL_PRECISION Mean: 0.497 | Std: 0.014 | 95% CI: [0.474, 0.519]

Early stopping at epoch 17. No improvement for 5 epochs.
Best epoch: 12 with kappa: 0.7706


## Test

In [15]:
model.load_state_dict(torch.load("models/resnet50.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS ===")
print(result)

100%|██████████| 796/796 [02:38<00:00,  5.02it/s]



=== TEST RESULTS ===
VAL_ACC      Mean: 52.856 | Std: 1.290 | 95% CI: [50.754, 54.965]
VAL_KAPPA    Mean: 0.759 | Std: 0.014 | 95% CI: [0.737, 0.783]
VAL_F1       Mean: 0.440 | Std: 0.013 | 95% CI: [0.419, 0.461]
VAL_RECALL   Mean: 0.440 | Std: 0.012 | 95% CI: [0.420, 0.460]
VAL_PRECISION Mean: 0.460 | Std: 0.014 | 95% CI: [0.436, 0.483]


## Comparison with EfficientNet-B0

In [16]:
print("\n" + "="*60)
print("ARCHITECTURE COMPARISON")
print("="*60)

# ResNet50 stats
resnet_params = sum(p.numel() for p in model.parameters())
print(f"\nResNet50:")
print(f"  Parameters: {resnet_params:,}")
print(f"  Test Kappa: {response[0]['kappa']['mean']:.4f}")
print(f"  Test Acc: {response[0]['acc']['mean']:.2f}%")

# EfficientNet-B0 reference (from logs)
print(f"\nEfficientNet-B0 (Reference):")
print(f"  Parameters: ~5,300,000")
print(f"  Test Kappa: 0.8260 (from logs)")
print(f"  Test Acc: 59.28%")

print("\n" + "="*60)


ARCHITECTURE COMPARISON

ResNet50:
  Parameters: 23,518,277


KeyError: 'kappa'